In [1]:
import numpy as np
import pandas as pd
import sys
import jieba
import pyLDAvis
import pyLDAvis.sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [2]:
## read documents
data = pd.read_csv("word_new.csv",header=None)
data.columns=["file","content"]

In [3]:
data.head()

,file,content
0,#名匠出品 必属精品#.docx,名匠出品 必属精品 2017-08-22 名匠出品 必属精品 面积 110...
1,#名匠出品 必属精品#[玫瑰] 面积：110平方.docx,名匠出品 必属精品 [玫瑰] 面积 110平方 2017-08-22 ...
2,1.docx,1 2018-04-15
3,108㎡优雅灰色装饰，考古、机械爱好者的家，真酷！.docx,108㎡优雅灰色装饰 考古 机械爱好者的家 真酷 2018-07-13 108㎡优...
4,10大主流装修风格详解，你喜欢哪个范？.docx,10大主流装修风格详解 你喜欢哪个范 2018-04-27 装修前总会纠结于风格的...


In [4]:
data.shape

(1218, 2)

In [5]:
def word_cut(text):
    return " ".join(jieba.cut(text))


In [6]:
data['cutted']=data['content'].apply(word_cut)

Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/sv/2x1mwq693cn_rg15vfcm7m6c0000gn/T/jieba.cache
Loading model cost 2.180 seconds.
Prefix dict has been built succesfully.


In [7]:
data[9:12]

,file,content,cutted
9,126㎡美法混搭四居，温馨又有格调的理想家！.docx,126㎡美法混搭四居 温馨又有格调的理想家 2017-11-10 你理想中的家是怎...,126 ㎡ 美法 混 搭 四居 温馨 又 有 格调 的 理想家 2017...
10,12月16号全天岳阳大酒店三楼舞龙厅.docx,12月16号全天岳阳大酒店三楼舞龙厅 2017-12-15 名匠装饰 感恩中国 |...,12 月 16 号 全天 岳阳 大酒店 三楼 舞龙 厅 2017 - 12 ...
11,12月16日全天岳阳大酒店三楼舞龙厅.docx,12月16日全天岳阳大酒店三楼舞龙厅 2017-12-15 名匠装饰 感恩中国 |...,12 月 16 日 全天 岳阳 大酒店 三楼 舞龙 厅 2017 - 12 ...


In [8]:
## Remove the empty character, transfer to list
cut_list=[]
for line in data['cutted']:
    t=line.split(' ')
    while '' in t:
        t.remove('')
    cut_list.append(t)

# f=open('cut.txt','w+',encoding="utf-8")
# for i in cut_list:
#     print(", ".join(i),file=f)
# f.close()

In [9]:
def removeStopWords(cut_list):
    cut_list_re = []
    cut_list_re_str = []
    stop_words = open('stopwords.txt',encoding="utf-8")
    stop_words_text = stop_words.read()
    stop_words.close()
    stop_words_list = stop_words_text.split('\n')
    
    for line in cut_list:
        t=[]
        for word in line:
            if word not in stop_words_list and word!=" " and len(word)!=1 and not word.isdigit() and word.isalpha():
                t.append(word)
        cut_list_re.append(t)

    # f = open('cut_re.txt', 'w',encoding="utf-8")
    for i in cut_list_re:
        cut_list_re_str.append(" ".join(i))
        # print(", ".join(i),file=f)
    # f.close()
    return cut_list_re, cut_list_re_str

In [10]:
(cut_list_re, cut_list_re_str)=removeStopWords(cut_list)

In [11]:
data.cutted=cut_list_re_str
data.cutted[:5]

0    出品 必属 精品 出品 必属 精品 面积 平方 小区 武广新 都城 风格 中式 中式 风格 ...
1                                    出品 必属 精品 玫瑰 面积 平方
2                                                     
3    优雅 灰色 装饰 考古 机械 爱好者 真酷 优雅 灰色 装饰 考古 机械 爱好者 真酷 本案...
4    主流 装修 风格 详解 喜欢 装修 总会 纠结 风格 选择 美式 欧式 地中海 中式 小伙伴...
Name: cutted, dtype: object

In [12]:
## Remove deleted articles
data=data[ ~ data['content'].str.contains(u'该内容已被发布者删除') ]
data=data.drop('content',axis=1)

In [13]:
data.shape
## 144 deleted articles

(1074, 2)

In [14]:
## Max size of cutted words is 4497
# max_len=0
# for index, row in data.iterrows():
#     max_len=max(max_len,len(row.cutted.split(" ")))
# print(max_len)

In [15]:
def remove_suffix(text):
    return text[:-5].strip(" ")

data['title']=data['file'].apply(remove_suffix)
data=data.drop('file',axis=1)
data=data[['title','cutted']]
data.to_csv("mingjiang_cutted_word.csv")

In [16]:
# Max size of cutted words is 4497
n_features = 1000
tf_vectorizer = CountVectorizer(strip_accents = 'unicode',
                                max_features=n_features,
                                max_df = 0.5,
                                min_df = 10)
tf = tf_vectorizer.fit_transform(data.cutted)
n_topics = 6
lda = LatentDirichletAllocation(n_topics=n_topics, max_iter=50,
                                learning_method='online',
                                learning_offset=50.,
                                random_state=0)

In [17]:
lda.fit(tf)

/anaconda3/lib/python3.7/site-packages/sklearn/decomposition/online_lda.py:314: DeprecationWarning: n_topics has been renamed to n_components in version 0.19 and will be removed in 0.21
  DeprecationWarning)


LatentDirichletAllocation(batch_size=128, doc_topic_prior=None,
             evaluate_every=-1, learning_decay=0.7,
             learning_method='online', learning_offset=50.0,
             max_doc_update_iter=100, max_iter=50, mean_change_tol=0.001,
             n_components=10, n_jobs=None, n_topics=6, perp_tol=0.1,
             random_state=0, topic_word_prior=None,
             total_samples=1000000.0, verbose=0)

In [18]:
topic_vec=lda.transform(tf)

In [19]:
topic_vec=pd.DataFrame(topic_vec[:,:])
topic_vec['Title']=data.title
column=[]
for topic_idx, topic in enumerate(lda.components_):
    column.append("Topic"+str(topic_idx))
column.append('Title')
topic_vec.columns=column
topic_vec.to_csv("topic_vec.csv")

In [20]:
def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print("Topic #%d:" % topic_idx)
        print(" ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]))
        print()


In [21]:
n_top_words = 10
tf_feature_names = tf_vectorizer.get_feature_names()
print_top_words(lda, tf_feature_names, n_top_words)

Topic #0:
装修 客户 公司 业主 设计 设计师 材料 施工 价格 时间

Topic #1:
装饰 工艺 业主 品质 感恩 设计 特权 工地 装修 服务

Topic #2:
客厅 厨房 卫生间 风水 摆放 鞋柜 装修 选择 卧室 影响

Topic #3:
阅读 分享 小区 装饰 插座 评论 热门 用于 户型 开关

Topic #4:
装修 瓷砖 材料 施工 安装 质量 环保 采用 墙面 涂料

Topic #5:
空间 风格 设计 生活 色彩 装修 装饰 家具 搭配 颜色



In [22]:
pyLDAvis.enable_notebook()
pyLDAvis.sklearn.prepare(lda, tf, tf_vectorizer)

/anaconda3/lib/python3.7/site-packages/pyLDAvis/_prepare.py:257: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  return pd.concat([default_term_info] + list(topic_dfs))


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.113849 -0.119119       1        1  27.311268
5     -0.121772  0.038545       2        1  25.343059
4     -0.054767 -0.155043       3        1  21.057654
1      0.225800 -0.071151       4        1  15.669485
2     -0.265616  0.015255       5        1   7.320776
3      0.102506  0.291514       6        1   3.297757, topic_info=     Category         Freq Term        Total  loglift  logprob
term                                                          
335   Default  1131.000000   客户  1131.000000  30.0000  30.0000
334   Default   480.000000   客厅   480.000000  29.0000  29.0000
843   Default  1364.000000   装饰  1364.000000  28.0000  28.0000
745   Default   947.000000   空间   947.000000  27.0000  27.0000
137   Default  1026.000000   公司  1026.000000  26.0000  26.0000
202   Default   425.000000   厨房   425.000000  25.0000  25.0000
379   Default   672.000000   工艺   672.000000  24.0000  24.0000
976   Default   772.000000   风格   772.000000  23.0000  23.0000
191   Default   387.000000  卫生间   387.000000  22.0000  22.0000
841   Default  3868.000000   装修  3868.000000  21.0000  21.0000
691   Default   511.000000   瓷砖   511.000000  20.0000  20.0000
356   Default   268.000000   小区   268.000000  19.0000  19.0000
39    Default  1238.000000   业主  1238.000000  18.0000  18.0000
693   Default   710.000000   生活   710.000000  17.0000  17.0000
977   Default   214.000000   风水   214.000000  16.0000  16.0000
945   Default   159.000000   阅读   159.000000  15.0000  15.0000
152   Default   168.000000   分享   168.000000  14.0000  14.0000
446   Default   381.000000   感恩   381.000000  13.0000  13.0000
243   Default   415.000000   品质   415.000000  12.0000  12.0000
494   Default   195.000000   摆放   195.000000  11.0000  11.0000
492   Default   165.000000   插座   165.000000  10.0000  10.0000
674   Default   272.000000   特权   272.000000   9.0000   9.0000
454   Default   241.000000   户型   241.000000   8.0000   8.0000
822   Default   374.000000   色彩   374.000000   7.0000   7.0000
912   Default   749.000000   选择   749.000000   6.0000   6.0000
190   Default   231.000000   卧室   231.000000   5.0000   5.0000
422   Default   410.000000   影响   410.000000   4.0000   4.0000
571   Default   959.000000   材料   959.000000   3.0000   3.0000
457   Default   299.000000   房间   299.000000   2.0000   2.0000
515   Default   204.000000   整装   204.000000   1.0000   1.0000
...       ...          ...  ...          ...      ...      ...
576    Topic6    36.372478  来美篇    37.211442   3.3891  -4.4711
352    Topic6    27.454951   导览    28.293762   3.3818  -4.7523
211    Topic6    37.962745   取消    40.940554   3.3364  -4.4283
869    Topic6    81.684666   评论    88.371471   3.3332  -3.6620
75     Topic6    44.958716   人气    49.375355   3.3182  -4.2591
808    Topic6    36.937451   联动    40.593645   3.3175  -4.4556
945    Topic6   144.664755   阅读   159.984038   3.3113  -3.0905
367    Topic6    41.631116   展开    46.159642   3.3087  -4.3360
550    Topic6    40.996602   最新    48.590483   3.2420  -4.3514
517    Topic6    48.201419   文章    58.177657   3.2238  -4.1895
470    Topic6    38.083565   投诉    46.134219   3.2202  -4.4251
766    Topic6    40.173007   精彩    51.118899   3.1710  -4.3717
791    Topic6    22.542917   绿化    28.976335   3.1609  -4.9495
152    Topic6   114.695190   分享   168.963403   3.0245  -3.3226
695    Topic6    57.072673   用于    97.459809   2.8768  -4.0205
998    Topic6    19.473419   鼓励    35.290036   2.8174  -5.0958
485    Topic6    41.362813   推荐    80.531204   2.7457  -4.3425
492    Topic6    81.799002   插座   165.323221   2.7083  -3.6606
138    Topic6    51.206762   关注   113.436934   2.6166  -4.1290
126    Topic6    18.053818   入户    40.838171   2.5957  -5.1715
906    Topic6    34.248656   进度    81.929306   2.5397  -4.5312
942    Topic6    33.589505   长沙    82.874161   2.5088  -4.5507
458    Topic6    14.122286   手机    35.367676   2.4939  -5.41